In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, make_scorer
import warnings
warnings.filterwarnings('ignore')

# Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Load data
train = pd.read_csv('./data/train.csv')
test = pd.read_csv('./data/test.csv')

# Define feature groups
numerical_features = ['model_year', 'milage']
categorical_features = ['brand', 'model', 'fuel_type', 'engine', 'transmission', 
                       'ext_col', 'int_col', 'accident']

# Define luxury and economy brands for feature engineering
luxury_brands = ['Porsche', 'Lamborghini', 'Ferrari', 'Rolls-Royce', 'Bentley', 'Aston', 'McLaren', 
                 'Maserati', 'Mercedes-Benz', 'BMW', 'Audi', 'Lexus', 'Cadillac', 'Land', 'Tesla']
economy_brands = ['Toyota', 'Honda', 'Nissan', 'Hyundai', 'Kia', 'Mazda', 'Subaru', 'Mitsubishi', 'FIAT']

# Select low cardinality features for one-hot encoding
high_cardinality_cats = ['model', 'engine']
low_cardinality_cats = [f for f in categorical_features if f not in high_cardinality_cats]

# Prepare training data
X_train = train.drop(['price', 'clean_title'], axis=1).copy()
y_train = train['price'].copy()

# Remove row with missing price
mask = y_train.notna()
X_train = X_train[mask]
y_train = y_train[mask]

# Feature engineering function
def create_features(df):
    df_fe = df.copy()
    
    # Fill missing values
    for col in categorical_features:
        if col in df_fe.columns:
            df_fe[col] = df_fe[col].fillna('missing')
    
    for col in numerical_features:
        if col in df_fe.columns:
            df_fe[col] = df_fe[col].fillna(df_fe[col].median())
    
    # Create new features
    df_fe['car_age'] = 2024 - df_fe['model_year']
    df_fe['mileage_per_year'] = df_fe['milage'] / (df_fe['car_age'] + 1)
    df_fe['is_luxury'] = df_fe['brand'].isin(luxury_brands).astype(int)
    df_fe['is_economy'] = df_fe['brand'].isin(economy_brands).astype(int)
    df_fe['has_accident'] = (df_fe['accident'] == 'At least 1 accident or damage reported').astype(int)
    df_fe['is_diesel'] = (df_fe['fuel_type'] == 'Diesel').astype(int)
    df_fe['high_mileage'] = (df_fe['milage'] > 100000).astype(int)
    
    return df_fe

# Apply feature engineering
X_train_fe = create_features(X_train)
X_test_fe = create_features(test)

# Define feature groups for modeling
numerical_features_fe = numerical_features + ['car_age', 'mileage_per_year']
binary_features = ['is_luxury', 'is_economy', 'has_accident', 'is_diesel', 'high_mileage']
features_to_use = numerical_features_fe + low_cardinality_cats + binary_features

# Select features
X_train_final = X_train_fe[features_to_use]
X_test_final = X_test_fe[features_to_use]

# Log transform target
y_train_log = np.log1p(y_train)

# Create preprocessing pipeline
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

binary_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0))
])

preprocessor = ColumnTransformer([
    ('num', numerical_transformer, numerical_features_fe),
    ('cat', categorical_transformer, low_cardinality_cats),
    ('bin', binary_transformer, binary_features)
])

# Create model pipeline
ridge_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=200, random_state=RANDOM_SEED))
])

# Define custom RMSE scorer that handles log transformation
def rmse_scorer(y_true, y_pred_log):
    y_pred = np.expm1(y_pred_log)  # Convert back from log
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(
    ridge_pipeline, X_train_final, y_train_log, 
    cv=cv, 
    scoring=make_scorer(rmse_scorer, greater_is_better=False),
    n_jobs=-1
)

# Show RMSE scores
cv_rmse_scores = -cv_scores
print(f"CV RMSE scores: {cv_rmse_scores}")
print(f"Mean CV RMSE: ${cv_rmse_scores.mean():,.2f}")
print(f"Std CV RMSE: ${cv_rmse_scores.std():,.2f}")

# Fit model on full training data
ridge_pipeline.fit(X_train_final, y_train_log)

# Make predictions on test set
test_predictions_log = ridge_pipeline.predict(X_test_final)
test_predictions = np.expm1(test_predictions_log)

# Create submission
submission = pd.DataFrame({
    'id': test['id'],
    'price': test_predictions
})

# Save submission
submission.to_csv('submission.csv', index=False)
print(f"\nSubmission saved with {len(submission)} predictions")